In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data, load_test_data

# --- 1. SETTINGS & PATHS ---
TRAIN_PATH = '../../datasets_challenge/train'
DIST_CSV = '../../datasets_challenge/sideSlip_distribution.csv'
MODEL_DIR = './expert_models'
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42) # Use your existing reproducibility util

In [2]:
class RNN(nn.Module):
    """
    Recurrent Neural Network for sideslip angle prediction.
    
    Architecture:
    1. RNN Layer - Processes sequential input
    2. Fully Connected Layer - Transforms RNN output
    3. Output Layer - Produces final prediction
    
    Dropout layers are included to prevent overfitting.
    """
    def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        
        # RNN layer: processes the sequence and captures temporal dependencies
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        
        # Fully connected layers for processing RNN output
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        
        # Dropout layers to prevent overfitting (randomly zeros out neurons during training)
        self.dropout1 = nn.Dropout(dropout[0])  # Applied to input
        self.dropout2 = nn.Dropout(dropout[1])  # Applied after RNN
        self.dropout3 = nn.Dropout(dropout[2])  # Applied after FC layer
        
        self.activation_fct = activation_fct
 
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        x = self.dropout1(x)
        
        # Process sequence through RNN
        x, _ = self.rnn_layer(x)  # Output shape: (batch_size, sequence_length, rnn_units)
        
        # Take only the last time step's output (t0)
        x = x[:, -1, :]  # Shape: (batch_size, rnn_units)
        
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))  # Apply fully connected layer with activation
        x = self.dropout3(x)
        x = self.output_layer(x)  # Final prediction
        
        return x

In [3]:
from sklearn.metrics import mean_absolute_error

device = "cuda" if torch.cuda.is_available() else "cpu"

model = RNN(
    input_size=22,          # number of features
    output_size=1,
    rnn_units=48,
    fc_units=48,
    dropout=[0.0, 0.0, 0.0]
).to(device)

criterion = nn.L1Loss()     # MAE (matches few-shot objective)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


In [4]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [5]:
def validate(model, dataloader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X).squeeze(1)
            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

    return mean_absolute_error(targets, preds)


In [6]:
print("Loading weighted train data...")
train_dataloader_weighted, val_loader, pipeline = load_data(
            train_path=TRAIN_PATH,
            dist_csv_path=DIST_CSV,
            val_file_name="train_part_017.csv", 
            columns_to_drop=["sideSlip", "vyCG", "time", "longitude","latitude"],
            sequence_length=20,
            threshold=True,
            threshold_value=3.0,
            threshold_column="vxCG",
            target='sideSlip',
            batchsize=64
        )

Loading weighted train data...
VALIDATION: Loading train_part_017.csv
Calculating weights for WeightedRandomSampler...


In [7]:
from utils.data_loading import load_data, load_test_data, feature_names

In [8]:
print("Loading normal train data...")
train_dataloader_uniform, val_dataloader, pipeline = load_data(
    train_path = "../../datasets_challenge/train",  # Path to training data
    train_ratio = 0.98,  # Use 98% for training, 2% for validation
    batchsize = 64,  # Number of samples per batch
    # columns_to_standardize = feature_names,  # Option to specify which columns to standardize
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],  # Remove these from features
    sequence_length = 20,  # Number of time steps in each sequence (t-19 to t0)
    threshold = True,  # Enable velocity thresholding
    threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
    threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
    target = "sideSlip",  # The variable we want to predict
    scaling = True,  # Apply standardization to features
    seed = 42,  # Random seed for train/val split
)

Loading normal train data...
../../datasets_challenge/train\train_part_001.csv
../../datasets_challenge/train\train_part_002.csv
../../datasets_challenge/train\train_part_003.csv
../../datasets_challenge/train\train_part_004.csv
../../datasets_challenge/train\train_part_005.csv
../../datasets_challenge/train\train_part_006.csv
../../datasets_challenge/train\train_part_007.csv
../../datasets_challenge/train\train_part_008.csv
../../datasets_challenge/train\train_part_009.csv
../../datasets_challenge/train\train_part_010.csv
../../datasets_challenge/train\train_part_011.csv
../../datasets_challenge/train\train_part_012.csv
../../datasets_challenge/train\train_part_013.csv
../../datasets_challenge/train\train_part_014.csv
../../datasets_challenge/train\train_part_015.csv
../../datasets_challenge/train\train_part_016.csv
../../datasets_challenge/train\train_part_017.csv
../../datasets_challenge/train\train_part_018.csv
../../datasets_challenge/train\train_part_019.csv
../../datasets_challe

In [9]:
CURRICULUM_EPOCHS = 3   # first 3 epochs = uniform
EPOCHS = 25


In [10]:
for epoch in range(EPOCHS):

    # SWITCH DATALOADER BASED ON EPOCH
    if epoch < CURRICULUM_EPOCHS:
        current_train_loader = train_dataloader_uniform
        phase = "UNIFORM"
    else:
        current_train_loader = train_dataloader_weighted
        phase = "WEIGHTED"

    # -------- TRAIN --------
    model.train()
    epoch_loss = 0.0

    for X, y in current_train_loader:
        X = X.to(device)
        y = y.to(device).unsqueeze(1)

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(current_train_loader)

    # -------- VALIDATE --------
    val_mae = validate(model, val_loader, device)

    print(
        f"Epoch {epoch+1:02d} [{phase}] | "
        f"Train Loss: {epoch_loss:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )


Epoch 01 [UNIFORM] | Train Loss: 0.2995 | Val MAE: 0.3138
Epoch 02 [UNIFORM] | Train Loss: 0.1958 | Val MAE: 0.3122
Epoch 03 [UNIFORM] | Train Loss: 0.1806 | Val MAE: 0.2965
Epoch 04 [WEIGHTED] | Train Loss: 0.2057 | Val MAE: 0.3284
Epoch 05 [WEIGHTED] | Train Loss: 0.1944 | Val MAE: 0.3219
Epoch 06 [WEIGHTED] | Train Loss: 0.1883 | Val MAE: 0.3408
Epoch 07 [WEIGHTED] | Train Loss: 0.1835 | Val MAE: 0.3697
Epoch 08 [WEIGHTED] | Train Loss: 0.1825 | Val MAE: 0.2865
Epoch 09 [WEIGHTED] | Train Loss: 0.1793 | Val MAE: 0.3008
Epoch 10 [WEIGHTED] | Train Loss: 0.1760 | Val MAE: 0.3011
Epoch 11 [WEIGHTED] | Train Loss: 0.1768 | Val MAE: 0.3216
Epoch 12 [WEIGHTED] | Train Loss: 0.1723 | Val MAE: 0.3662
Epoch 13 [WEIGHTED] | Train Loss: 0.1706 | Val MAE: 0.2741
Epoch 14 [WEIGHTED] | Train Loss: 0.1701 | Val MAE: 0.3486
Epoch 15 [WEIGHTED] | Train Loss: 0.1689 | Val MAE: 0.3101
Epoch 16 [WEIGHTED] | Train Loss: 0.1674 | Val MAE: 0.4041
Epoch 17 [WEIGHTED] | Train Loss: 0.1662 | Val MAE: 0.3507


In [14]:
# Save best model (important for snapshot / ensemble)
best_val_mae = float("inf")
best_model_state = None
if val_mae < best_val_mae:
    best_val_mae = val_mae
    best_model_state = model.state_dict()
model.load_state_dict(best_model_state)
print(f"✓ Best validation MAE: {best_val_mae:.4f}")

✓ Best validation MAE: 0.3260


In [15]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [16]:
# Generate predictions on test set
model.eval()
total_loss = 0  
test_preds = []
test_ground_truths = []  # Will contain placeholder values (all zeros)

with torch.no_grad():
    for step, (x, y) in enumerate(test_dataloader):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = criterion(pred, y)
        # Note: loss is meaningless here since y contains placeholder values
        
        # Store predictions
        test_preds.append(pred.reshape(-1).cpu().numpy())
        test_ground_truths.append(y.reshape(-1).cpu().numpy())

    # Concatenate all batches
    test_ground_truths = np.concatenate(test_ground_truths)
    test_preds = np.concatenate(test_preds)
    
print("Shape of Predictions", test_preds.shape)

Shape of Predictions (49328,)


In [17]:
# Save predictions in the required format for submission
y_pred = pd.DataFrame({"predicted_sideslip": test_preds})
y_pred["id"] = range(len(test_preds))  # Add sample IDs (0 to 49327)

# Save to CSV file
y_pred.to_csv("predictions09_02.csv", index=False)

print(f"✓ Predictions saved to 'predictions09_02.csv'")
print(f"✓ Total samples: {len(test_preds)}")
print(f"✓ Ready for submission to the competition!")

✓ Predictions saved to 'predictions09_02.csv'
✓ Total samples: 49328
✓ Ready for submission to the competition!
